### DSC630-T303 Predictive Analytics - 10.2 Assignment: Recommender System
Scope of this project is to create a recommender system that allows users to input a movie they like (in the data set) and recommends ten other movies for them to watch.

Data Source: https://grouplens.org/datasets/movielens/


References: 
https://medium.com/data-science/introduction-to-recommender-systems-6c66cf15ada

https://towardsdatascience.com/movie-recommendation-system-based-on-movielens-ef0df580cd0e/

https://www.datacamp.com/tutorial/recommender-systems-python

https://alicege005.medium.com/creating-a-simple-recommendation-system-based-on-tf-idf-and-counter-vectorizer-b3c91aa3aff0


In [56]:
# Import required libraries
import pandas as pd
import numpy as np

# TF-IDF is used to convert genre text into numerical vectors.
from sklearn.feature_extraction.text import TfidfVectorizer

# Cosine similarity is used to compare movies.
from sklearn.metrics.pairwise import cosine_similarity

## csr_matrix creates a sparse rating matrix
from scipy.sparse import csr_matrix



In [57]:
#Load the Data

movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")

# Display the first few rows
print("Movies dataset:")
display(movies.head())



Movies dataset:


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [58]:
print("Ratings dataset:")
display(ratings.head())

Ratings dataset:


,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976
3,1,30,5.0,944249077
4,1,32,5.0,943228858


In [59]:
# Check dataset size
print("Movies shape:", movies.shape)
print("Ratings shape:", ratings.shape)



Movies shape: (87585, 3)
Ratings shape: (32000204, 4)


In [60]:
# Check missing values
print("\nMissing values in movies:")
print(movies.isnull().sum())

print("\nMissing values in ratings:")
print(ratings.isnull().sum())


Missing values in movies:
movieId    0
title      0
genres     0
dtype: int64

Missing values in ratings:
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64


In [61]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [62]:
print("Movies rows:", movies.shape[0])
print("Ratings rows:", ratings.shape[0])
print("Unique users:", ratings["userId"].nunique())
print("Unique movies in ratings:", ratings["movieId"].nunique())

Movies rows: 87585
Ratings rows: 32000204
Unique users: 200948
Unique movies in ratings: 84432


#### Clean and standardize columns

In [63]:
# Clean and standardize the dataset


# Keep only needed columns from movies
movies = movies[["movieId", "title", "genres"]].copy()

# Keep only needed columns from ratings
# timestamp is not needed for this recommender system
ratings = ratings[["userId", "movieId", "rating"]].copy()

# Convert ID and rating columns to numeric
movies["movieId"] = pd.to_numeric(movies["movieId"], errors="coerce")
ratings["userId"] = pd.to_numeric(ratings["userId"], errors="coerce")
ratings["movieId"] = pd.to_numeric(ratings["movieId"], errors="coerce")
ratings["rating"] = pd.to_numeric(ratings["rating"], errors="coerce")

# Drop rows with missing important values
movies = movies.dropna(subset=["movieId", "title", "genres"])
ratings = ratings.dropna(subset=["userId", "movieId", "rating"])

# Convert ID columns to integers
movies["movieId"] = movies["movieId"].astype(int)
ratings["userId"] = ratings["userId"].astype(int)
ratings["movieId"] = ratings["movieId"].astype(int)

print("Clean movies shape:", movies.shape)
print("Clean ratings shape:", ratings.shape)

display(movies.head())
display(ratings.head())

Clean movies shape: (87585, 3)
Clean ratings shape: (32000204, 3)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


,userId,movieId,rating
0,1,17,4.0
1,1,25,1.0
2,1,29,2.0
3,1,30,5.0
4,1,32,5.0


TF-IDF (Term Frequency–Inverse Document Frequency) changes text into numbers so the computer can compare text.TF-IDF works with text, and separating the genres with spaces allows each genre to be treated as an individual word. A new column called genres_clean can be created so the original genre information would still be preserved.

In [65]:
# Clean genres column for TF-IDF
movies["genres_clean"] = movies["genres"].str.replace("|", " ", regex=False)

display(movies[["movieId", "title", "genres", "genres_clean"]].head())

,movieId,title,genres,genres_clean
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Adventure Animation Children Comedy Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy,Adventure Children Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance,Comedy Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Comedy Drama Romance
4,5,Father of the Bride Part II (1995),Comedy,Comedy


In [66]:
# Check dataset size


print("Number of movies in movies.csv:", movies["movieId"].nunique())
print("Number of ratings:", ratings.shape[0])
print("Number of unique users:", ratings["userId"].nunique())
print("Number of unique rated movies:", ratings["movieId"].nunique())

Number of movies in movies.csv: 87585
Number of ratings: 32000204
Number of unique users: 200948
Number of unique rated movies: 84432


#### Filter the large ratings dataset

In [67]:
# Filter the large dataset


# Count how many ratings each movie received
movie_rating_counts = (
    ratings.groupby("movieId")
    .size()
    .reset_index(name="rating_count")
)

# Keep movies with at least 100 ratings
popular_movies = movie_rating_counts[
    movie_rating_counts["rating_count"] >= 100
]["movieId"]

ratings_filtered = ratings[
    ratings["movieId"].isin(popular_movies)
].copy()

# Count how many movies each user rated
user_rating_counts = (
    ratings_filtered.groupby("userId")
    .size()
    .reset_index(name="user_rating_count")
)

# Keep users who rated at least 20 movies
active_users = user_rating_counts[
    user_rating_counts["user_rating_count"] >= 20
]["userId"]

ratings_filtered = ratings_filtered[
    ratings_filtered["userId"].isin(active_users)
].copy()

print("Original ratings shape:", ratings.shape)
print("Filtered ratings shape:", ratings_filtered.shape)
print("Filtered unique users:", ratings_filtered["userId"].nunique())
print("Filtered unique movies:", ratings_filtered["movieId"].nunique())

Original ratings shape: (32000204, 3)
Filtered ratings shape: (31217906, 3)
Filtered unique users: 200322
Filtered unique movies: 12191


#### Create sparse rating matrix

 csr_matrix creates a sparse rating matrix
 
This is useful because most users rate only a small number of movies, so the movie-user rating table has many empty cells.

In [68]:
# Convert movieId and userId into row/column positions
movie_categories = pd.Categorical(ratings_filtered["movieId"])
user_categories = pd.Categorical(ratings_filtered["userId"])

movie_codes = movie_categories.codes
user_codes = user_categories.codes

# Create sparse movie-user rating matrix
rating_matrix = csr_matrix(
    (
        ratings_filtered["rating"],
        (movie_codes, user_codes)
    )
)

print("Sparse rating matrix shape:", rating_matrix.shape)

Sparse rating matrix shape: (12191, 200322)


#### Create movies_model

In [69]:
# Create movies_model in the same order as rating_matrix

# The recommender uses both rating similarity and genre similarity, 
# The rating_matrix rows are movies, so movies_model must contain the same movies in the same order as the rating_matrix
# This alignment is important because the movie index will be used later to compare ratings and genres correctly


# Get the movieIds used in the sparse rating matrix
movie_ids_in_matrix = list(movie_categories.categories)

# Keep only movies that are present in the sparse rating matrix
movies_model = movies[movies["movieId"].isin(movie_ids_in_matrix)].copy()

# Reorder movies_model to match the row order of rating_matrix
movies_model = (
    movies_model
    .set_index("movieId")
    .loc[movie_ids_in_matrix]
    .reset_index()
)

print("Movies model shape:", movies_model.shape)
display(movies_model.head())

Movies model shape: (12191, 4)


,movieId,title,genres,genres_clean
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Adventure Animation Children Comedy Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy,Adventure Children Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance,Comedy Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Comedy Drama Romance
4,5,Father of the Bride Part II (1995),Comedy,Comedy


In [33]:
display(movies_model[["movieId", "title", "genres", "genres_clean"]].head())

,movieId,title,genres,genres_clean
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Adventure Animation Children Comedy Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy,Adventure Children Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance,Comedy Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Comedy Drama Romance
4,5,Father of the Bride Part II (1995),Comedy,Comedy


In [71]:
#Vectorization

# The genres_clean column contains text values such as:"Adventure Animation Children Comedy Fantasy"
# Since machine learning calculations cannot directly compare text,TF-IDF is used to convert the genre words into numerical values.
# Each movie becomes a row, and each genre word becomes a column.these numerical values are later used to calculate genre similarity


tfidf = TfidfVectorizer(stop_words="english")
# Apply TF-IDF to the cleaned genre text.
# The result is stored in genre_matrix.
# genre_matrix will be used later in the recommendation function.

genre_matrix = tfidf.fit_transform(movies_model["genres_clean"])


print("Genre TF-IDF matrix shape:", genre_matrix.shape)

Genre TF-IDF matrix shape: (12191, 23)


In [72]:
# Show the genre words/features created by TF-IDF
tfidf_features = tfidf.get_feature_names_out()

print(tfidf_features)

['action' 'adventure' 'animation' 'children' 'comedy' 'crime'
 'documentary' 'drama' 'fantasy' 'fi' 'film' 'genres' 'horror' 'imax'
 'listed' 'musical' 'mystery' 'noir' 'romance' 'sci' 'thriller' 'war'
 'western']


In [73]:
# Convert first 5 rows of TF-IDF matrix into a dataframe just to see show how each movie is represented numerically based on its genres
tfidf_sample = pd.DataFrame(
    genre_matrix[:5].toarray(),
    columns=tfidf.get_feature_names_out()
)

# Add movie titles for easier understanding
tfidf_sample.insert(0, "title", movies_model["title"].head(5).values)

display(tfidf_sample)

,title,action,adventure,animation,children,comedy,crime,documentary,drama,fantasy,...,imax,listed,musical,mystery,noir,romance,sci,thriller,war,western
0,Toy Story (1995),0.0,0.418924,0.516676,0.499194,0.276075,0.0,0.0,0.000000,0.481805,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
1,Jumanji (1995),0.0,0.516904,0.000000,0.615947,0.000000,0.0,0.0,0.000000,0.594492,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
2,Grumpier Old Men (1995),0.0,0.000000,0.000000,0.000000,0.585657,0.0,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.810559,0.0,0.0,0.0,0.0
3,Waiting to Exhale (1995),0.0,0.000000,0.000000,0.000000,0.524393,0.0,0.0,0.445275,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.725770,0.0,0.0,0.0,0.0
4,Father of the Bride Part II (1995),0.0,0.000000,0.000000,0.000000,1.000000,0.0,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0


#### Add average rating and number of ratings

In [74]:
# Add average rating and number of ratings


# Remove old rating summary columns if this cell was already run before
movies_model = movies_model.drop(
    columns=["average_rating", "number_of_ratings"],
    errors="ignore"
)

# Calculate average rating and rating count for each movie
rating_summary = (
    ratings_filtered.groupby("movieId")
    .agg(
        average_rating=("rating", "mean"),
        number_of_ratings=("rating", "count")
    )
    .reset_index()
)

# Merge rating summary with movies_model
movies_model = movies_model.merge(
    rating_summary,
    on="movieId",
    how="left"
)

# Fill missing values with 0
movies_model["average_rating"] = movies_model["average_rating"].fillna(0)
movies_model["number_of_ratings"] = movies_model["number_of_ratings"].fillna(0)

display(
    movies_model[
        ["movieId", "title", "genres", "average_rating", "number_of_ratings"]
    ].head()
)

,movieId,title,genres,average_rating,number_of_ratings
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.897450,68971
1,2,Jumanji (1995),Adventure|Children|Fantasy,3.275821,28901
2,3,Grumpier Old Men (1995),Comedy|Romance,3.139462,13129
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,2.845331,2806
4,5,Father of the Bride Part II (1995),Comedy,3.059392,13150


In [75]:
movies_model.columns

Index(['movieId', 'title', 'genres', 'genres_clean', 'average_rating',
       'number_of_ratings'],
      dtype='object')

In [77]:
# Add average rating and number of ratings safely

# Remove old summary columns if they already exist
movies_model = movies_model.drop(
    columns=["average_rating", "number_of_ratings"],
    errors="ignore"
)

rating_summary = (
    ratings_filtered.groupby("movieId")
    .agg(
        average_rating=("rating", "mean"),
        number_of_ratings=("rating", "count")
    )
    .reset_index()
)

movies_model = movies_model.merge(
    rating_summary,
    on="movieId",
    how="left"
)

movies_model["average_rating"] = movies_model["average_rating"].fillna(0)
movies_model["number_of_ratings"] = movies_model["number_of_ratings"].fillna(0)

display(
    movies_model[
        ["movieId", "title", "genres", "genres_clean", "average_rating", "number_of_ratings"]
    ].head()
)

,movieId,title,genres,genres_clean,average_rating,number_of_ratings
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Adventure Animation Children Comedy Fantasy,3.897450,68971
1,2,Jumanji (1995),Adventure|Children|Fantasy,Adventure Children Fantasy,3.275821,28901
2,3,Grumpier Old Men (1995),Comedy|Romance,Comedy Romance,3.139462,13129
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Comedy Drama Romance,2.845331,2806
4,5,Father of the Bride Part II (1995),Comedy,Comedy,3.059392,13150


#### Create movie search function

In [78]:
# Search function to find exact movie title

def search_movie_title(keyword):
    """
    Search movie titles by keyword.
    This helps find the exact MovieLens title.
    """

    results = movies_model[
        movies_model["title"].str.contains(keyword, case=False, na=False)
    ]

    return results[
        ["movieId", "title", "genres", "average_rating", "number_of_ratings"]
    ].head(20)

In [79]:
search_movie_title("Toy Story")

,movieId,title,genres,average_rating,number_of_ratings
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.897450,68971
2628,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,3.812114,32674
8691,78499,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,3.827608,20320
9730,106022,Toy Story of Terror (2013),Animation|Children|Comedy,3.342618,359
10022,115875,Toy Story Toons: Hawaiian Vacation (2011),Adventure|Animation|Children|Comedy|Fantasy,3.043165,139
10023,115879,Toy Story Toons: Small Fry (2011),Adventure|Animation|Children|Comedy|Fantasy,3.077236,123
10121,120474,Toy Story That Time Forgot (2014),Animation|Children,3.216617,337
11575,201588,Toy Story 4 (2019),Adventure|Animation|Children|Comedy,3.588897,2801


In [80]:
search_movie_title("Matrix")

,movieId,title,genres,average_rating,number_of_ratings
2158,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller,4.156604,93692
5020,6365,"Matrix Reloaded, The (2003)",Action|Adventure|Sci-Fi|Thriller|IMAX,3.372654,29832
5369,6934,"Matrix Revolutions, The (2003)",Action|Adventure|Sci-Fi|Thriller|IMAX,3.236074,23696
6630,27660,"Animatrix, The (2003)",Action|Animation|Drama|Sci-Fi,3.654511,5265
11021,172255,The Matrix Revisited (2001),Documentary,3.624183,153
12016,263009,The Matrix Resurrections (2021),Action|Adventure|Sci-Fi,2.713870,1204


#### Create the recommendation function
This function builds the final hybrid recommender system. It first checks whether the movie entered by the user exists in the dataset. Then it calculates rating similarity using the sparse movie-user rating matrix and genre similarity using the TF-IDF genre matrix. The final recommendation score combines 70% rating similarity and 30% genre similarity. The input movie itself is removed, and the ten movies with the highest hybrid similarity scores are returned as recommendations.

In [81]:
# Hybrid Movie Recommendation Function
## This function recommends 10 movies similar to the movie entered by the user
# This function uses a hybrid recommendation method:
# 1. Rating similarity: compares movies based on user rating patterns.
# 2. Genre similarity: compares movies based on genre text using TF-IDF.
####
# Final hybrid score:
# 70% rating similarity + 30% genre similarity


def recommend_movies(movie_title, num_recommendations=10, rating_weight=0.70, genre_weight=0.30):
    """
    Recommend ten movies similar to the movie entered by the user.

    Hybrid method:
    - 70% rating similarity
    - 30% genre similarity

    Rating similarity compares movies using user rating patterns.
    Genre similarity compares movies using TF-IDF genre vectors.
    """

    # Check whether the entered movie exists
    if movie_title not in movies_model["title"].values:
        return f"Movie '{movie_title}' was not found. Use search_movie_title() to find the exact title."

    # Find the row index of the selected movie
    movie_index = movies_model[movies_model["title"] == movie_title].index[0]

    # Rating similarity
    rating_similarity_scores = cosine_similarity(
        rating_matrix[movie_index],
        rating_matrix
    ).flatten()

    # Genre similarity   TF-IDF is used in the genre_matrix
    genre_similarity_scores = cosine_similarity(
        genre_matrix[movie_index],
        genre_matrix
    ).flatten()

    # Hybrid similarity score
    hybrid_scores = (
        rating_weight * rating_similarity_scores +
        genre_weight * genre_similarity_scores
    )

    # Create results dataframe
    results = movies_model.copy()

    results["rating_similarity"] = rating_similarity_scores
    results["genre_similarity"] = genre_similarity_scores
    results["hybrid_similarity"] = hybrid_scores

    # Remove the input movie itself
    results = results[results["title"] != movie_title]

    # Sort by final recommendation score
    results = results.sort_values(
        by="hybrid_similarity",
        ascending=False
    )

    # Return top 10 recommendations
    recommendations = results[
        [
            "title",
            "genres",
            "average_rating",
            "number_of_ratings",
            "rating_similarity",
            "genre_similarity",
            "hybrid_similarity"
        ]
    ].head(num_recommendations)

    return recommendations

#### Test the recommender

In [82]:
search_movie_title("Toy Story")

,movieId,title,genres,average_rating,number_of_ratings
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.897450,68971
2628,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,3.812114,32674
8691,78499,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,3.827608,20320
9730,106022,Toy Story of Terror (2013),Animation|Children|Comedy,3.342618,359
10022,115875,Toy Story Toons: Hawaiian Vacation (2011),Adventure|Animation|Children|Comedy|Fantasy,3.043165,139
10023,115879,Toy Story Toons: Small Fry (2011),Adventure|Animation|Children|Comedy|Fantasy,3.077236,123
10121,120474,Toy Story That Time Forgot (2014),Animation|Children,3.216617,337
11575,201588,Toy Story 4 (2019),Adventure|Animation|Children|Comedy,3.588897,2801


In [83]:
recommend_movies("Toy Story (1995)", 10)

,title,genres,average_rating,number_of_ratings,rating_similarity,genre_similarity,hybrid_similarity
2628,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,3.812114,32674,0.574967,1.000000,0.702477
4048,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,3.837422,46024,0.513296,1.000000,0.659307
3587,Shrek (2001),Adventure|Animation|Children|Comedy|Fantasy|Ro...,3.740493,54827,0.514554,0.934133,0.640428
5029,Finding Nemo (2003),Adventure|Animation|Children|Comedy,3.816449,46118,0.500563,0.876278,0.613277
1958,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy,3.557962,26733,0.466452,0.876278,0.589400
541,Aladdin (1992),Adventure|Animation|Children|Comedy|Musical,3.707249,50437,0.526777,0.726551,0.586709
6235,"Incredibles, The (2004)",Action|Adventure|Animation|Children|Comedy,3.848232,41458,0.474530,0.807494,0.574419
338,"Lion King, The (1994)",Adventure|Animation|Children|Drama|Musical|IMAX,3.833036,51511,0.536861,0.545717,0.539518
1904,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy,3.213230,12850,0.326213,1.000000,0.528349
2518,Who Framed Roger Rabbit? (1988),Adventure|Animation|Children|Comedy|Crime|Fant...,3.551849,25449,0.394310,0.837050,0.527132


In [84]:
search_movie_title("Matrix")

,movieId,title,genres,average_rating,number_of_ratings
2158,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller,4.156604,93692
5020,6365,"Matrix Reloaded, The (2003)",Action|Adventure|Sci-Fi|Thriller|IMAX,3.372654,29832
5369,6934,"Matrix Revolutions, The (2003)",Action|Adventure|Sci-Fi|Thriller|IMAX,3.236074,23696
6630,27660,"Animatrix, The (2003)",Action|Animation|Drama|Sci-Fi,3.654511,5265
11021,172255,The Matrix Revisited (2001),Documentary,3.624183,153
12016,263009,The Matrix Resurrections (2021),Action|Adventure|Sci-Fi,2.713870,1204


In [85]:
recommend_movies("Matrix, The (1999)", 10)

,title,genres,average_rating,number_of_ratings,rating_similarity,genre_similarity,hybrid_similarity
1045,"Terminator, The (1984)",Action|Sci-Fi|Thriller,3.902037,47125,0.573229,1.000000,0.701261
1004,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Sci-Fi,4.130794,72075,0.658263,0.786384,0.696699
239,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,4.100247,84915,0.640289,0.786384,0.684117
503,Blade Runner (1982),Action|Sci-Fi|Thriller,4.109959,46281,0.530061,1.000000,0.671042
542,Terminator 2: Judgment Day (1991),Action|Sci-Fi,3.961950,68370,0.562719,0.904726,0.665321
1017,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Sci-Fi,3.991734,67441,0.595810,0.786384,0.652982
1320,Men in Black (a.k.a. MIB) (1997),Action|Comedy|Sci-Fi,3.593816,48430,0.569297,0.847150,0.652653
443,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller,3.698559,75207,0.515735,0.889145,0.627758
4371,Spider-Man (2002),Action|Adventure|Sci-Fi|Thriller,3.479338,38622,0.510276,0.889145,0.623936
2493,Fight Club (1999),Action|Crime|Drama|Thriller,4.228811,77256,0.693869,0.452134,0.621348


#### User Input Version

In [92]:
# User input version


user_movie = input("Enter a movie you like from the MovieLens dataset: ")

recommendations = recommend_movies(user_movie, 10)

display(recommendations)

Enter a movie you like from the MovieLens dataset:  Terminator, The (1984)


,title,genres,average_rating,number_of_ratings,rating_similarity,genre_similarity,hybrid_similarity
542,Terminator 2: Judgment Day (1991),Action|Sci-Fi,3.961950,68370,0.685766,0.904726,0.751454
503,Blade Runner (1982),Action|Sci-Fi|Thriller,4.109959,46281,0.585041,1.000000,0.709529
2158,"Matrix, The (1999)",Action|Sci-Fi|Thriller,4.156604,93692,0.573229,1.000000,0.701261
2946,Predator (1987),Action|Sci-Fi|Thriller,3.672484,20773,0.551433,1.000000,0.686003
1008,Aliens (1986),Action|Adventure|Horror|Sci-Fi,4.007261,38839,0.668274,0.701720,0.678307
2456,Total Recall (1990),Action|Adventure|Sci-Fi|Thriller,3.628682,26919,0.586049,0.889145,0.676978
1004,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Sci-Fi,4.130794,72075,0.621615,0.786384,0.671045
1320,Men in Black (a.k.a. MIB) (1997),Action|Comedy|Sci-Fi,3.593816,48430,0.573760,0.847150,0.655777
1021,Alien (1979),Horror|Sci-Fi,4.068388,45205,0.655744,0.649470,0.653862
1017,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Sci-Fi,3.991734,67441,0.596116,0.786384,0.653196


#### Wrapper function to focus on the title

In [95]:
# Wrapper function to display only 10 movie names


def recommend_movie_names(user_input, num_recommendations=10):
    """
    Takes a partial or full movie name from the user
    and displays only 10 recommended movie names.
    """

    # Remove extra spaces from user input
    user_input = user_input.strip()

    # First check exact title match
    exact_matches = movies_model[
        movies_model["title"].str.lower() == user_input.lower()
    ]

    if not exact_matches.empty:
        exact_title = exact_matches.iloc[0]["title"]
    else:
        # Search using title and title_clean
        # regex=False treats user input as normal text, not a regex pattern
        matches = movies_model[
            movies_model["title"].str.contains(user_input, case=False, na=False, regex=False) |
            movies_model["title_clean"].str.contains(user_input, case=False, na=False, regex=False)
        ]

        if matches.empty:
            print(f"No movie found for '{user_input}'. Try another keyword.")
            return

        exact_title = matches.iloc[0]["title"]

    print(f"Movie selected: {exact_title}")
    print("\nTop 10 Recommended Movies:\n")

    recommendations = recommend_movies(exact_title, num_recommendations)

    movie_names = recommendations["title"].reset_index(drop=True)

    for i, movie in enumerate(movie_names, start=1):
        print(f"{i}. {movie}")

In [96]:
user_movie = input("Enter a movie you like from the MovieLens dataset: ")

recommend_movie_names(user_movie, 10)

Enter a movie you like from the MovieLens dataset:  Toy Story (1995)


Movie selected: Toy Story (1995)

Top 10 Recommended Movies:

1. Toy Story 2 (1999)
2. Monsters, Inc. (2001)
3. Shrek (2001)
4. Finding Nemo (2003)
5. Bug's Life, A (1998)
6. Aladdin (1992)
7. Incredibles, The (2004)
8. Lion King, The (1994)
9. Antz (1998)
10. Who Framed Roger Rabbit? (1988)


In [97]:
user_movie = input("Enter a movie you like from the MovieLens dataset: ")

recommend_movie_names(user_movie, 10)

Enter a movie you like from the MovieLens dataset:  Jumanji 


Movie selected: Jumanji (1995)

Top 10 Recommended Movies:

1. Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)
2. Casper (1995)
3. Toy Story (1995)
4. NeverEnding Story, The (1984)
5. Indian in the Cupboard, The (1995)
6. Home Alone (1990)
7. Chronicles of Narnia: The Lion, the Witch and the Wardrobe, The (2005)
8. Addams Family Values (1993)
9. Honey, I Shrunk the Kids (1989)
10. Monsters, Inc. (2001)


In [98]:
user_movie = input("Enter a movie you like from the MovieLens dataset: ")

recommend_movie_names(user_movie, 10)

Enter a movie you like from the MovieLens dataset:  Matrix


Movie selected: Matrix, The (1999)

Top 10 Recommended Movies:

1. Terminator, The (1984)
2. Star Wars: Episode V - The Empire Strikes Back (1980)
3. Star Wars: Episode IV - A New Hope (1977)
4. Blade Runner (1982)
5. Terminator 2: Judgment Day (1991)
6. Star Wars: Episode VI - Return of the Jedi (1983)
7. Men in Black (a.k.a. MIB) (1997)
8. Jurassic Park (1993)
9. Spider-Man (2002)
10. Fight Club (1999)


In [99]:
user_movie_namesearch= input("Enter a movie you like from the MovieLens dataset: ")

recommend_movie_names(user_movie_namesearch, 10)

Enter a movie you like from the MovieLens dataset:  Star Wars


Movie selected: Star Wars: Episode IV - A New Hope (1977)

Top 10 Recommended Movies:

1. Star Wars: Episode V - The Empire Strikes Back (1980)
2. Star Wars: Episode VI - Return of the Jedi (1983)
3. Raiders of the Lost Ark (Indiana Jones and the Raiders of the Lost Ark) (1981)
4. Matrix, The (1999)
5. Star Wars: Episode I - The Phantom Menace (1999)
6. Back to the Future (1985)
7. Jurassic Park (1993)
8. Terminator 2: Judgment Day (1991)
9. Aliens (1986)
10. Independence Day (a.k.a. ID4) (1996)


#### Collaborative filtering is used in the recommendation function when the system calculates rating_similarity_scores from the rating_matrix, because this compares movies based on user rating patterns from ratings.csv. Content-based filtering is used when the system calculates genre_similarity_scores from the genre_matrix, because this compares movies based on genre information from movies.csv after the genres were cleaned and converted into numerical values using TF-IDF. The final recommendation score combines both methods by using 70% collaborative filtering score and 30% content-based filtering score, so the system recommends movies that are similar based on both user behavior and movie content.